In [ ]:
# In[1]:

import pandas as pd
import pytz

# Reuse timezone variable
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into DataFrames and keep them in kernel variables
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to convert epoch seconds to Asia/Shanghai localized datetimes
def to_shanghai_iso(epoch_series):
    ts = pd.to_datetime(epoch_series.astype('int64'), unit='s', utc=True)
    return ts.dt.tz_convert(tz)

# Convert timestamp columns to timezone-aware datetimes in Asia/Shanghai and store as new columns
df_metric['_ts_sh'] = to_shanghai_iso(df_metric['timestamp'])
df_trace['_ts_sh'] = to_shanghai_iso(df_trace['timestamp'])
df_log['_ts_sh'] = to_shanghai_iso(df_log['timestamp'])
df_error['_ts_sh'] = to_shanghai_iso(df_error['timestamp'])

# metric.csv summary
metric_total = df_metric.shape[0]
metric_cmdbs = list(pd.Series(df_metric['cmdb_id'].unique()).astype(str))[:20]
metric_kpis = list(pd.Series(df_metric['kpi_name'].unique()).astype(str))[:20]
metric_min_ts = df_metric['_ts_sh'].min()
metric_max_ts = df_metric['_ts_sh'].max()

metric_summary = pd.Series({
    "metric.csv - total_rows": f"{metric_total}",
    "metric.csv - distinct_cmdb_id_top20": ", ".join(metric_cmdbs) if metric_cmdbs else "",
    "metric.csv - distinct_kpi_name_top20": ", ".join(metric_kpis) if metric_kpis else "",
    "metric.csv - timestamp_range_Asia/Shanghai": f"min={metric_min_ts.isoformat() if pd.notnull(metric_min_ts) else 'N/A'}, max={metric_max_ts.isoformat() if pd.notnull(metric_max_ts) else 'N/A'}"
})

# trace.csv summary
trace_total = df_trace.shape[0]
trace_cmdbs = list(pd.Series(df_trace['cmdb_id'].unique()).astype(str))[:20]
trace_names = list(pd.Series(df_trace['trace_name'].unique()).astype(str))[:20]
trace_min_ts = df_trace['_ts_sh'].min()
trace_max_ts = df_trace['_ts_sh'].max()

trace_summary = pd.Series({
    "trace.csv - total_rows": f"{trace_total}",
    "trace.csv - distinct_cmdb_id_top20": ", ".join(trace_cmdbs) if trace_cmdbs else "",
    "trace.csv - distinct_trace_name_top20": ", ".join(trace_names) if trace_names else "",
    "trace.csv - timestamp_range_Asia/Shanghai": f"min={trace_min_ts.isoformat() if pd.notnull(trace_min_ts) else 'N/A'}, max={trace_max_ts.isoformat() if pd.notnull(trace_max_ts) else 'N/A'}"
})

# log.csv summary
log_total = df_log.shape[0]
log_cmdbs = list(pd.Series(df_log['cmdb_id'].unique()).astype(str))[:20]
log_names = list(pd.Series(df_log['log_name'].unique()).astype(str))[:20]
log_min_ts = df_log['_ts_sh'].min()
log_max_ts = df_log['_ts_sh'].max()

log_summary = pd.Series({
    "log.csv - total_rows": f"{log_total}",
    "log.csv - distinct_cmdb_id_top20": ", ".join(log_cmdbs) if log_cmdbs else "",
    "log.csv - distinct_log_name_top20": ", ".join(log_names) if log_names else "",
    "log.csv - timestamp_range_Asia/Shanghai": f"min={log_min_ts.isoformat() if pd.notnull(log_min_ts) else 'N/A'}, max={log_max_ts.isoformat() if pd.notnull(log_max_ts) else 'N/A'}"
})

# error_logs.csv summary (with up to 5 sample messages)
error_total = df_error.shape[0]
error_cmdbs = list(pd.Series(df_error['cmdb_id'].unique()).astype(str))[:20]
error_min_ts = df_error['_ts_sh'].min()
error_max_ts = df_error['_ts_sh'].max()

error_summary = pd.Series({
    "error_logs.csv - total_rows": f"{error_total}",
    "error_logs.csv - distinct_cmdb_id_top20": ", ".join(error_cmdbs) if error_cmdbs else "",
    "error_logs.csv - timestamp_range_Asia/Shanghai": f"min={error_min_ts.isoformat() if pd.notnull(error_min_ts) else 'N/A'}, max={error_max_ts.isoformat() if pd.notnull(error_max_ts) else 'N/A'}"
})

# Prepare up to 5 sample messages with timestamp (Asia/Shanghai), cmdb_id, and truncated message (<=200 chars)
if error_total > 0:
    error_samples = df_error[['timestamp', 'cmdb_id', 'message', '_ts_sh']].copy()
    error_samples = error_samples.sort_values('timestamp', ascending=True).head(5)
    error_samples['message_trunc'] = error_samples['message'].astype(str).str.slice(0, 200)
    # Format timestamp via apply to use datetime.isoformat()
    error_samples['ts_sh_iso'] = error_samples['_ts_sh'].apply(lambda x: x.isoformat() if pd.notnull(x) else '')
    error_samples = error_samples.loc[:, ['ts_sh_iso', 'cmdb_id', 'message_trunc']]
    error_samples.columns = ['timestamp_Asia/Shanghai', 'cmdb_id', 'message_sample']
else:
    error_samples = pd.DataFrame(columns=['timestamp_Asia/Shanghai', 'cmdb_id', 'message_sample'])

# Return compact summaries (variables) for IPython display
metric_summary, trace_summary, log_summary, error_summary, error_samples

```
Out[1]:
```
```python
# Summarize the previously computed summaries in plain English as a single compact string.
summary_lines = []

summary_lines.append(f"metric.csv: total rows = {metric_summary['metric.csv - total_rows']}.")
summary_lines.append(f"  Distinct cmdb_id (top up to 20) = {metric_summary['metric.csv - distinct_cmdb_id_top20']}.")
summary_lines.append(f"  Distinct kpi_name (top up to 20) = {metric_summary['metric.csv - distinct_kpi_name_top20']}.")
summary_lines.append(f"  Timestamp range (Asia/Shanghai) = {metric_summary['metric.csv - timestamp_range_Asia/Shanghai']}.")

summary_lines.append(f"trace.csv: total rows = {trace_summary['trace.csv - total_rows']}.")
summary_lines.append(f"  Distinct cmdb_id (top up to 20) = {trace_summary['trace.csv - distinct_cmdb_id_top20']}.")
summary_lines.append(f"  Distinct trace_name (top up to 20) = {trace_summary['trace.csv - distinct_trace_name_top20']}.")
summary_lines.append(f"  Timestamp range (Asia/Shanghai) = {trace_summary['trace.csv - timestamp_range_Asia/Shanghai']}.")

summary_lines.append(f"log.csv: total rows = {log_summary['log.csv - total_rows']}.")
summary_lines.append(f"  Distinct cmdb_id (top up to 20) = {log_summary['log.csv - distinct_cmdb_id_top20']}.")
summary_lines.append(f"  Distinct log_name (top up to 20) = {log_summary['log.csv - distinct_log_name_top20']}.")
summary_lines.append(f"  Timestamp range (Asia/Shanghai) = {log_summary['log.csv - timestamp_range_Asia/Shanghai']}.")

summary_lines.append(f"error_logs.csv: total rows = {error_summary['error_logs.csv - total_rows']}.")
summary_lines.append(f"  Distinct cmdb_id (top up to 20) = {error_summary['error_logs.csv - distinct_cmdb_id_top20']}.")
summary_lines.append(f"  Timestamp range (Asia/Shanghai) = {error_summary['error_logs.csv - timestamp_range_Asia/Shanghai']}.")

# Add up to 5 sample error messages (already prepared in error_samples)
if not error_samples.empty:
    summary_lines.append("  Sample error log entries (up to 5):")
    for _, row in error_samples.iterrows():
        ts = row['timestamp_Asia/Shanghai']
        cmdb = row['cmdb_id']
        msg = row['message_sample']
        summary_lines.append(f"    {ts} - {cmdb} - {msg}")
else:
    summary_lines.append("  No error log samples available.")

# Join into one compact plain-English string
summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(metric.csv - total_rows                                                                  147972
metric.csv - distinct_cmdb_id_top20           adservice, adservice-0, adservice-1, adservice...
metric.csv - distinct_kpi_name_top20          app.grpc.count, app.grpc.mrt, app.grpc.rr, app...
metric.csv - timestamp_range_Asia/Shanghai    min=2022-03-21T10:30:00+08:00, max=2022-03-21T...
dtype: object, trace.csv - total_rows                                                                   36152
trace.csv - distinct_cmdb_id_top20           adservice-0, adservice-1, adservice-2, adservi...
trace.csv - distinct_trace_name_top20        trace.from_frontend-0.duration_mean, trace.fro...
trace.csv - timestamp_range_Asia/Shanghai    min=2022-03-21T10:30:00+08:00, max=2022-03-21T...
dtype: object, log.csv - total_rows                                                                    1814
log.csv - distinct_cmdb_id_top20           adservice-0, adservice-1, adservice-2, cartser...
log.csv - distinct_log_name_top20                             log.error_count, log.row_count
log.csv - timestamp_range_Asia/Shanghai    min=2022-03-21T10:30:00+08:00, max=2022-03-21T...
dtype: object, error_logs.csv - total_rows                                                                    1552
error_logs.csv - distinct_cmdb_id_top20                          frontend-2, frontend-1, frontend-0
error_logs.csv - timestamp_range_Asia/Shanghai    min=2022-03-21T10:30:00+08:00, max=2022-03-21T...
dtype: object,      timestamp_Asia/Shanghai     cmdb_id                                     message_sample
0  2022-03-21T10:30:00+08:00  frontend-2  severity: warning, message: failed to retrieve...
1  2022-03-21T10:30:00+08:00  frontend-1  severity: warning, message: failed to retrieve...
2  2022-03-21T10:30:02+08:00  frontend-0  severity: warning, message: failed to retrieve...
3  2022-03-21T10:30:02+08:00  frontend-2  severity: warning, message: failed to retrieve...
4  2022-03-21T10:30:03+08:00  frontend-1  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse existing df_metric and tz, and _ts_sh column
# 1) compute global P95 per (cmdb_id, kpi_name) using all rows
p95_df = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95)
p95_df = p95_df.rename(columns={'value': 'p95_threshold'})

# 2) filter metric rows to the incident window (Asia/Shanghai)
start = pd.Timestamp('2022-03-21T10:30:00', tz=tz)
end = pd.Timestamp('2022-03-21T11:00:00', tz=tz)
window_df = df_metric[(df_metric['_ts_sh'] >= start) & (df_metric['_ts_sh'] < end)].copy()

# 3) merge window rows with p95 thresholds
merged = window_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# 3) aggregate per pair
def agg_group(g):
    total_points = len(g)
    max_value = g['value'].max() if total_points>0 else np.nan
    # anomaly rows where value >= p95_threshold (p95_threshold is same for group)
    p95 = g['p95_threshold'].iloc[0] if not g['p95_threshold'].isnull().all() else np.nan
    if np.isnan(p95):
        anomaly_count = 0
        earliest_anom_ts = pd.NaT
    else:
        mask = g['value'] >= p95
        anomaly_count = int(mask.sum())
        if anomaly_count > 0:
            earliest_anom_ts = g.loc[mask, '_ts_sh'].min()
        else:
            earliest_anom_ts = pd.NaT
    # compute max_over_threshold_pct only when anomaly_count>0 and p95>0
    if anomaly_count > 0 and p95 != 0 and not np.isnan(p95):
        max_over_pct = 100.0 * (max_value - p95) / p95
    else:
        max_over_pct = np.nan
    return pd.Series({
        'p95_threshold': p95,
        'total_points_in_window': total_points,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp_in_window': earliest_anom_ts,
        'max_value_in_window': max_value,
        'max_over_threshold_pct': max_over_pct
    })

agg = merged.groupby(['cmdb_id', 'kpi_name'], as_index=False).apply(lambda g: agg_group(g)).reset_index(drop=True)

# Keep full intermediate aggregated results in kernel variable for follow-ups
agg_metric_p95_window = agg.copy()

# 4) filter to anomaly_count_in_window > 0, format timestamp, sort and limit top 20
res = agg_metric_p95_window[agg_metric_p95_window['anomaly_count_in_window'] > 0].copy()

# format earliest timestamp to ISO strings in Asia/Shanghai, keep NaT as None
def fmt_ts(x):
    if pd.isna(x) or x is pd.NaT:
        return None
    return pd.Timestamp(x).tz_convert(tz).isoformat()

res['earliest_anomaly_timestamp_in_window'] = res['earliest_anomaly_timestamp_in_window'].apply(fmt_ts)

# sort by anomaly_count desc, then max_over_threshold_pct desc
res = res.sort_values(['anomaly_count_in_window', 'max_over_threshold_pct'], ascending=[False, False])

# select and order columns as requested
res = res.loc[:, ['cmdb_id', 'kpi_name', 'p95_threshold', 'total_points_in_window',
                  'anomaly_count_in_window', 'earliest_anomaly_timestamp_in_window',
                  'max_value_in_window', 'max_over_threshold_pct']]

# limit to top 20 rows for compact display
res_top20 = res.head(20)

# Display the compact result (and keep agg_metric_p95_window in kernel)
res_top20, agg_metric_p95_window.shape

```
Out[2]:
```
- Computed global P95 for 4,942 distinct (cmdb_id, kpi_name) pairs and evaluated the incident window 2022-03-21T10:30:00+08:00 (inclusive) to 2022-03-21T11:00:00+08:00 (exclusive).  
- Returned the top 20 (cmdb_id, kpi_name) pairs with anomalies (anomaly_count_in_window > 0), sorted by anomaly count then max_over_threshold_pct.  
- Observation: every top-20 entry has anomaly_count_in_window = 30 (i.e., all 30 points in the 30-minute window are ≥ the global P95), and earliest anomaly timestamps are at or immediately after the window start (2022-03-21T10:30:00+08:00).  
- Leading KPI types in the top results: container.node-*.container_sockets, container.node-*.container_file_descriptors, and node-level system.udp.connect.num.  
- Example highlights from the top rows:  
  - checkoutservice2-0 / container.node-5.container_sockets — p95 = 3.0, total_points = 30, anomalies = 30, earliest = 2022-03-21T10:30:00+08:00, max = 3.5, max_over_threshold_pct ≈ 16.67%.  
  - productcatalogservice2-0 / container.node-6.container_sockets — p95 = 3.0, total = 30, anomalies = 30, max = 3.5, max_over_threshold_pct ≈ 16.67%.  
  - productcatalogservice-0 / container.node-5.container_sockets — p95 = 4.0, total = 30, anomalies = 30, max = 4.5, max_over_threshold_pct = 12.5%.  
  - node-2 / system.udp.connect.num — p95 = 10.0, total = 30, anomalies = 30, max = 11.0, max_over_threshold_pct = 10.0%.  
  - Several JVM/service metrics appear with anomalies but very small over-threshold percentages (≈0–1%).  
- Full aggregated results are stored in the kernel variable agg_metric_p95_window for any follow-up analysis or drilling down.

The original code execution output of IPython Kernel is also provided below for reference:

(                       cmdb_id                                           kpi_name  p95_threshold  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  max_over_threshold_pct
1840        checkoutservice2-0                 container.node-5.container_sockets            3.0                      30                       30            2022-03-21T10:30:00+08:00             3.500000               16.666667
4214  productcatalogservice2-0                 container.node-6.container_sockets            3.0                      30                       30            2022-03-21T10:30:00+08:00             3.500000               16.666667
3866   productcatalogservice-0                 container.node-5.container_sockets            4.0                      30                       30            2022-03-21T10:30:00+08:00             4.500000               12.500000
3982   productcatalogservice-1                 container.node-5.container_sockets            4.0                      30                       30            2022-03-21T10:30:00+08:00             4.500000               12.500000
3300                    node-2                             system.udp.connect.num           10.0                      30                       30            2022-03-21T10:30:00+08:00            11.000000               10.000000
980               adservice2-0                 container.node-6.container_sockets            8.0                      30                       30            2022-03-21T10:30:00+08:00             8.666667                8.333333
1799        checkoutservice2-0        container.node-5.container_file_descriptors            9.0                      30                       30            2022-03-21T10:30:00+08:00             9.500000                5.555556
4173  productcatalogservice2-0        container.node-6.container_file_descriptors            9.0                      30                       30            2022-03-21T10:30:00+08:00             9.500000                5.555556
3825   productcatalogservice-0        container.node-5.container_file_descriptors           10.0                      30                       30            2022-03-21T10:30:00+08:00            10.500000                5.000000
3941   productcatalogservice-1        container.node-5.container_file_descriptors           10.0                      30                       30            2022-03-21T10:30:00+08:00            10.500000                5.000000
337                  adservice                 runtime.jvm_threads_state.RUNNABLE            5.0                      30                       30            2022-03-21T10:30:00+08:00             5.083333                1.666667
833                 adservice2  runtime.java_lang_OperatingSystem_OpenFileDesc...           95.0                      30                       30            2022-03-21T10:30:00+08:00            95.750000                0.789474
2                    adservice                                        app.grpc.rr          100.0                      30                       30            2022-03-21T10:30:00+08:00           100.000000                0.000000
4                    adservice                                     app.http.count           24.0                      30                       30            2022-03-21T10:30:00+08:00            24.000000                0.000000
6                    adservice                                        app.http.rr          100.0                      30                       30            2022-03-21T10:30:00+08:00           100.000000                0.000000
7                    adservice                                        app.http.sr          100.0                      30                       30            2022-03-21T10:30:00+08:00           100.000000                0.000000
8                    adservice    runtime.java_lang_ClassLoading_LoadedClassCount         4990.0                      30                       30            2022-03-21T10:30:00+08:00          4990.000000                0.000000
9                    adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...         4990.0                      30                       30            2022-03-21T10:30:00+08:00          4990.000000                0.000000
12                   adservice  runtime.java_lang_Compilation_CompilationTimeM...            1.0                      30                       30            2022-03-21T10:30:00+08:00             1.000000                0.000000
15                   adservice  runtime.java_lang_GarbageCollector_CollectionC...            3.0                      30                       30            2022-03-21T10:30:00+08:00             3.000000                0.000000, (4942, 8))```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse df_trace, tz, and _ts_sh from kernel

# 1) compute global P95 per (cmdb_id, trace_name) using all rows
p95_trace = df_trace.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].quantile(0.95)
p95_trace = p95_trace.rename(columns={'value': 'p95_threshold'})

# 2) filter trace rows to the incident window (Asia/Shanghai)
start = pd.Timestamp('2022-03-21T10:30:00', tz=tz)
end = pd.Timestamp('2022-03-21T11:00:00', tz=tz)
window_trace = df_trace[(df_trace['_ts_sh'] >= start) & (df_trace['_ts_sh'] < end)].copy()

# 3) merge window rows with p95 thresholds
merged_t = window_trace.merge(p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# aggregation function per group
def agg_group_trace(g):
    total_points = len(g)
    max_value = g['value'].max() if total_points > 0 else np.nan
    p95 = g['p95_threshold'].iloc[0] if not g['p95_threshold'].isnull().all() else np.nan
    if np.isnan(p95):
        anomaly_count = 0
        earliest_anom_ts = pd.NaT
    else:
        mask = g['value'] >= p95
        anomaly_count = int(mask.sum())
        earliest_anom_ts = g.loc[mask, '_ts_sh'].min() if anomaly_count > 0 else pd.NaT
    if anomaly_count > 0 and p95 not in (0, np.nan):
        max_over_pct = 100.0 * (max_value - p95) / p95
    else:
        max_over_pct = np.nan
    return pd.Series({
        'p95_threshold': p95,
        'total_points_in_window': total_points,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp_in_window': earliest_anom_ts,
        'max_value_in_window': max_value,
        'max_over_threshold_pct': max_over_pct
    })

# apply aggregation grouped by (cmdb_id, trace_name)
agg_trace = merged_t.groupby(['cmdb_id', 'trace_name']).apply(agg_group_trace).reset_index()

# store full aggregated results for follow-up
agg_trace_p95_window = agg_trace.copy()

# 4) filter to anomaly_count_in_window > 0, format timestamp, sort and limit top 20
res_t = agg_trace_p95_window[agg_trace_p95_window['anomaly_count_in_window'] > 0].copy()

def fmt_ts(x):
    if pd.isna(x) or x is pd.NaT:
        return None
    return pd.Timestamp(x).tz_convert(tz).isoformat()

res_t['earliest_anomaly_timestamp_in_window'] = res_t['earliest_anomaly_timestamp_in_window'].apply(fmt_ts)

res_t = res_t.sort_values(['anomaly_count_in_window', 'max_over_threshold_pct'], ascending=[False, False])

res_t = res_t.loc[:, ['cmdb_id', 'trace_name', 'p95_threshold', 'total_points_in_window',
                      'anomaly_count_in_window', 'earliest_anomaly_timestamp_in_window',
                      'max_value_in_window', 'max_over_threshold_pct']]

res_top20_trace = res_t.head(20)

# Display compact result and keep full aggregation in kernel
res_top20_trace, agg_trace_p95_window.shape

```
Out[3]:
```
```python
# Compose a compact plain-English summary of the trace P95 analysis and display as a single string.
summary_lines = []
summary_lines.append("Trace P95 analysis summary:")
summary_lines.append("- Computed global P95 for 1,232 distinct (cmdb_id, trace_name) pairs and evaluated the incident window 2022-03-21T10:30:00+08:00 (inclusive) to 2022-03-21T11:00:00+08:00 (exclusive).")
summary_lines.append("- Returned the top pairs with anomalies (anomaly_count_in_window > 0), limited to 20 rows for compactness.")
summary_lines.append("- Many top entries show anomaly_count_in_window = 30 (all 30 points in the 30-minute window >= global P95) with earliest anomalies at the window start (2022-03-21T10:30:00+08:00).")
summary_lines.append("- Several duration-related traces (e.g., trace.*.duration_p95 / duration_mean) appear with p95 = 0.001 and full-window anomalies (e.g., cartservice2-0 trace.from_frontend2-0.duration_p95 and frontend2-0 trace.to_cartservice2-0.duration_p95).")
summary_lines.append("- Many error_rate traces have p95 = 0.000 and window values = 0.000, producing anomaly counts = 30 but max_over_threshold_pct is NaN or not meaningful due to zero threshold.")
summary_lines.append("- Top examples from the result set:")
summary_lines.append("  • cartservice2-0 / trace.from_frontend2-0.duration_p95 — p95=0.001, total=30, anomalies=30, earliest=2022-03-21T10:30:00+08:00, max=0.001.")
summary_lines.append("  • frontend2-0 / trace.to_cartservice2-0.duration_p95 — p95=0.001, total=30, anomalies=30, earliest=2022-03-21T10:30:00+08:00, max=0.001.")
summary_lines.append("  • Multiple adservice/* and cartservice/* traces for error_rate and duration_* show full-window anomalies but often with p95=0.0 (so percentage over threshold is not meaningful).")
summary_lines.append("- The full aggregated trace-P95 results are stored in the kernel variable 'agg_trace_p95_window' for follow-up drilling or checks.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                                  trace_name  p95_threshold  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  max_over_threshold_pct
129  cartservice2-0         trace.from_frontend2-0.duration_p95          0.001                      30                       30            2022-03-21T10:30:00+08:00                0.001                     0.0
857     frontend2-0        trace.to_cartservice2-0.duration_p95          0.001                      30                       30            2022-03-21T10:30:00+08:00                0.001                     0.0
2       adservice-0            trace.from_frontend-0.error_rate          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
6       adservice-0            trace.from_frontend-1.error_rate          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
10      adservice-0            trace.from_frontend-2.error_rate          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
14      adservice-1            trace.from_frontend-0.error_rate          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
18      adservice-1            trace.from_frontend-1.error_rate          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
22      adservice-1            trace.from_frontend-2.error_rate          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
26      adservice-2            trace.from_frontend-0.error_rate          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
30      adservice-2            trace.from_frontend-1.error_rate          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
34      adservice-2            trace.from_frontend-2.error_rate          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
38     adservice2-0           trace.from_frontend2-0.error_rate          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
40    cartservice-0  trace.from_checkoutservice-0.duration_mean          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
41    cartservice-0   trace.from_checkoutservice-0.duration_p95          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
42    cartservice-0     trace.from_checkoutservice-0.error_rate          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
44    cartservice-0  trace.from_checkoutservice-1.duration_mean          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
45    cartservice-0   trace.from_checkoutservice-1.duration_p95          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
46    cartservice-0     trace.from_checkoutservice-1.error_rate          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
48    cartservice-0  trace.from_checkoutservice-2.duration_mean          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN
49    cartservice-0   trace.from_checkoutservice-2.duration_p95          0.000                      30                       30            2022-03-21T10:30:00+08:00                0.000                     NaN, (1232, 8))```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd

# Reuse df_log, df_error, tz, and their _ts_sh columns from kernel

# 1) Compute global P95 per (cmdb_id, log_name) using all rows (global threshold)
p95_log = df_log.groupby(['cmdb_id', 'log_name'], as_index=False)['value'].quantile(0.95)
p95_log = p95_log.rename(columns={'value': 'p95_threshold'})

# 2) Filter log rows to the incident window (Asia/Shanghai)
start = pd.Timestamp('2022-03-21T10:30:00', tz=tz)
end = pd.Timestamp('2022-03-21T11:00:00', tz=tz)
window_log = df_log[(df_log['_ts_sh'] >= start) & (df_log['_ts_sh'] < end)].copy()

# 3) Merge window rows with p95 thresholds
merged_log = window_log.merge(p95_log, on=['cmdb_id', 'log_name'], how='left')

# Aggregation per (cmdb_id, log_name)
def agg_group_log(g):
    total_points = len(g)
    max_value = g['value'].max() if total_points > 0 else np.nan
    p95 = g['p95_threshold'].iloc[0] if not g['p95_threshold'].isnull().all() else np.nan
    if np.isnan(p95):
        anomaly_count = 0
        earliest_anom_ts = pd.NaT
    else:
        mask = g['value'] >= p95
        anomaly_count = int(mask.sum())
        earliest_anom_ts = g.loc[mask, '_ts_sh'].min() if anomaly_count > 0 else pd.NaT
    if anomaly_count > 0 and p95 not in (0, np.nan):
        max_over_pct = 100.0 * (max_value - p95) / p95
    else:
        max_over_pct = np.nan
    return pd.Series({
        'p95_threshold': p95,
        'total_points_in_window': total_points,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp_in_window': earliest_anom_ts,
        'max_value_in_window': max_value,
        'max_over_threshold_pct': max_over_pct
    })

agg_log = merged_log.groupby(['cmdb_id', 'log_name']).apply(agg_group_log).reset_index()

# Store full aggregated results for follow-up
agg_log_p95_window = agg_log.copy()

# 4) Filter to anomaly_count_in_window > 0, format timestamp, sort and limit top 20
res_log = agg_log_p95_window[agg_log_p95_window['anomaly_count_in_window'] > 0].copy()

def fmt_ts(x):
    if pd.isna(x) or x is pd.NaT:
        return None
    return pd.Timestamp(x).tz_convert(tz).isoformat()

res_log['earliest_anomaly_timestamp_in_window'] = res_log['earliest_anomaly_timestamp_in_window'].apply(fmt_ts)

res_log = res_log.sort_values(['anomaly_count_in_window', 'max_over_threshold_pct'], ascending=[False, False])

res_log = res_log.loc[:, ['cmdb_id', 'log_name', 'p95_threshold', 'total_points_in_window',
                          'anomaly_count_in_window', 'earliest_anomaly_timestamp_in_window',
                          'max_value_in_window', 'max_over_threshold_pct']]

res_top20_log = res_log.head(20)

# -------------------------
# Part 2: error_logs analysis
# -------------------------
# Filter error logs to the incident window
window_error = df_error[(df_error['_ts_sh'] >= start) & (df_error['_ts_sh'] < end)].copy()

# Aggregate per cmdb_id: message_count_in_window and earliest_message_timestamp_in_window
agg_error = window_error.groupby('cmdb_id', as_index=False).agg(
    message_count_in_window=('message', 'size'),
    earliest_message_timestamp_in_window=('_ts_sh', 'min')
)

# Format earliest timestamp to ISO or None
agg_error['earliest_message_timestamp_in_window'] = agg_error['earliest_message_timestamp_in_window'].apply(fmt_ts)

# Sort and take top 20 cmdb_ids by message_count_in_window
agg_error = agg_error.sort_values('message_count_in_window', ascending=False).reset_index(drop=True)
agg_error_summary = agg_error.copy()  # store full aggregated error_logs summary for follow-up
err_top20 = agg_error_summary.head(20)

# For top 3 cmdb_ids, collect up to 5 sample raw messages with their timestamps (Asia/Shanghai)
top3_cmdbs = err_top20['cmdb_id'].head(3).tolist()
if top3_cmdbs:
    samples = window_error[window_error['cmdb_id'].isin(top3_cmdbs)].copy()
    # sort by timestamp and take up to 5 per cmdb_id
    samples = samples.sort_values(['cmdb_id', 'timestamp'], ascending=[True, True])
    samples = samples.groupby('cmdb_id').head(5).reset_index(drop=True)
    # format timestamp and truncate message for compactness
    samples['timestamp_Asia/Shanghai'] = samples['_ts_sh'].apply(fmt_ts)
    samples['message_sample'] = samples['message'].astype(str).str.slice(0, 300)
    err_samples_top3 = samples.loc[:, ['cmdb_id', 'timestamp_Asia/Shanghai', 'message_sample']]
else:
    err_samples_top3 = pd.DataFrame(columns=['cmdb_id', 'timestamp_Asia/Shanghai', 'message_sample'])

# Return compact results: top log anomalies, top error counts, and samples for top3 error cmdb_ids
res_top20_log, err_top20, err_samples_top3, agg_log_p95_window.shape, agg_error_summary.shape

```
Out[4]:
```
```python
# Compose a compact plain-English summary using prior results stored in kernel variables:
summary_lines = []

# Log P95 results summary
summary_lines.append("Logs (log.csv) P95 analysis:")
summary_lines.append("- Computed global P95 per (cmdb_id, log_name) and inspected window 2022-03-21T10:30:00+08:00 to 2022-03-21T11:00:00+08:00.")
summary_lines.append("- Many services show anomalies for log.error_count where p95_threshold = 0.0 and all 30 points in-window >= p95 (anomaly_count = 30); earliest anomaly timestamps ≈ 2022-03-21T10:30:00+08:00.")
summary_lines.append("- Example top log anomalies (service / log_name / p95 / total_points / anomalies / earliest / max):")
# include up to 5 examples from res_top20_log if available
if 'res_top20_log' in globals() and not res_top20_log.empty:
    for _, r in res_top20_log.head(5).iterrows():
        summary_lines.append(f"  • {r['cmdb_id']} / {r['log_name']} / p95={r['p95_threshold']} / total={r['total_points_in_window']} / anomalies={r['anomaly_count_in_window']} / earliest={r['earliest_anomaly_timestamp_in_window']} / max={r['max_value_in_window']}")
else:
    summary_lines.append("  • (no log anomaly rows found)")

# Error logs summary
summary_lines.append("Error logs (error_logs.csv) in-window summary:")
if 'err_top20' in globals() and not err_top20.empty:
    # top counts
    for i, row in err_top20.head(3).iterrows():
        summary_lines.append(f"  {i+1}. {row['cmdb_id']} — message_count_in_window={row['message_count_in_window']}, earliest_message={row['earliest_message_timestamp_in_window']}")
    # include note about others
    if len(err_top20) > 3:
        summary_lines.append(f"  (other top components up to 20 included; showing top 3 above)")
else:
    summary_lines.append("  No error log messages in the incident window.")

# For the top 3 cmdb_ids, show up to 5 sample messages each (compact)
if 'err_samples_top3' in globals() and not err_samples_top3.empty:
    summary_lines.append("Sample error messages for top 3 cmdb_ids (up to 5 each):")
    for cmdb in err_top20['cmdb_id'].head(3).tolist():
        samp = err_samples_top3[err_samples_top3['cmdb_id'] == cmdb]
        if samp.empty:
            continue
        summary_lines.append(f"  - {cmdb}:")
        for _, s in samp.iterrows():
            # truncate message sample to keep line short
            msg = s['message_sample']
            summary_lines.append(f"      {s['timestamp_Asia/Shanghai']} - {msg}")
else:
    summary_lines.append("  No sample error messages available.")

summary_lines.append("- Full aggregated results saved in kernel variables: 'agg_log_p95_window' and 'agg_error_summary' for follow-up analysis.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id         log_name  p95_threshold  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  max_over_threshold_pct
0               adservice-0  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
2               adservice-1  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
4               adservice-2  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
6             cartservice-0  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
8             cartservice-1  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
10            cartservice-2  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
12        checkoutservice-0  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
14        checkoutservice-1  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
16        checkoutservice-2  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
18        currencyservice-0  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
20        currencyservice-1  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
22        currencyservice-2  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
24           emailservice-0  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
26           emailservice-1  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
28           emailservice-2  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
36         paymentservice-0  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
38         paymentservice-1  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
40         paymentservice-2  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
42  productcatalogservice-0  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN
44  productcatalogservice-1  log.error_count            0.0                      30                       30            2022-03-21T10:30:00+08:00                  0.0                     NaN,       cmdb_id  message_count_in_window earliest_message_timestamp_in_window
0  frontend-0                      535            2022-03-21T10:30:02+08:00
1  frontend-2                      527            2022-03-21T10:30:00+08:00
2  frontend-1                      490            2022-03-21T10:30:00+08:00,        cmdb_id    timestamp_Asia/Shanghai                                     message_sample
0   frontend-0  2022-03-21T10:30:02+08:00  severity: warning, message: failed to retrieve...
1   frontend-0  2022-03-21T10:30:06+08:00  severity: warning, message: failed to retrieve...
2   frontend-0  2022-03-21T10:30:08+08:00  severity: warning, message: failed to retrieve...
3   frontend-0  2022-03-21T10:30:14+08:00  severity: warning, message: failed to retrieve...
4   frontend-0  2022-03-21T10:30:20+08:00  severity: warning, message: failed to retrieve...
5   frontend-1  2022-03-21T10:30:00+08:00  severity: warning, message: failed to retrieve...
6   frontend-1  2022-03-21T10:30:03+08:00  severity: warning, message: failed to retrieve...
7   frontend-1  2022-03-21T10:30:07+08:00  severity: warning, message: failed to retrieve...
8   frontend-1  2022-03-21T10:30:09+08:00  severity: warning, message: failed to retrieve...
9   frontend-1  2022-03-21T10:30:13+08:00  severity: warning, message: failed to retrieve...
10  frontend-2  2022-03-21T10:30:00+08:00  severity: warning, message: failed to retrieve...
11  frontend-2  2022-03-21T10:30:02+08:00  severity: warning, message: failed to retrieve...
12  frontend-2  2022-03-21T10:30:04+08:00  severity: warning, message: failed to retrieve...
13  frontend-2  2022-03-21T10:30:09+08:00  severity: warning, message: failed to retrieve...
14  frontend-2  2022-03-21T10:30:12+08:00  severity: warning, message: failed to retrieve..., (62, 8), (3, 3))```
```

In [ ]:
# In[5]:

import re
import pandas as pd
import numpy as np
import pytz

# Reuse timezone
tz = pytz.timezone('Asia/Shanghai')

# Reuse aggregated DataFrames from kernel: agg_trace_p95_window, agg_metric_p95_window, agg_error_summary
# Defensive copies
trace_agg = agg_trace_p95_window.copy()
metric_agg = agg_metric_p95_window.copy()

# Ensure earliest timestamps in metric_agg are normalized to pandas Timestamps (if present as strings)
def parse_maybe_iso(x):
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, pd.Timestamp):
        return x
    try:
        return pd.to_datetime(x).tz_convert(tz) if pd.to_datetime(x).tz is not None else pd.to_datetime(x).tz_localize(tz)
    except Exception:
        try:
            # fallback parse naive then localize
            return pd.to_datetime(x).tz_localize(tz)
        except Exception:
            return pd.NaT

if 'earliest_anomaly_timestamp_in_window' in metric_agg.columns:
    metric_agg['_earliest_ts_parsed'] = metric_agg['earliest_anomaly_timestamp_in_window'].apply(parse_maybe_iso)
else:
    metric_agg['_earliest_ts_parsed'] = pd.NaT

# 1) Select frontend-origin traces: trace_name containing from_frontend, from_frontend2, or starting with 'trace.to_'
is_frontend_origin = trace_agg['trace_name'].str.contains(r'from_frontend|from_frontend2|^trace\.to_', regex=True, na=False)
front_traces = trace_agg[is_frontend_origin].copy()

# 2) Extract downstream target component
def extract_downstream(trace_name, trace_cmdb):
    # Prefer explicit 'to_<component>' pattern if present
    m = re.search(r'to_([A-Za-z0-9\-]+)', trace_name)
    if m:
        return m.group(1)
    # If 'from_frontend' pattern: assume the trace row's cmdb_id is the downstream target
    if 'from_frontend' in trace_name or 'from_frontend2' in trace_name:
        return trace_cmdb
    # fallback: try to find a service-like token after first dot
    parts = trace_name.split('.')
    if len(parts) > 1:
        # try to find token that looks like service-pod (contains letters and a dash or digits)
        for p in parts[1:]:
            m2 = re.search(r'([a-z]+[a-z0-9\-]*\d*\-\d+)', p)
            if m2:
                return m2.group(1)
        # otherwise return the second part as candidate
        return parts[1]
    return None

front_traces['downstream_component'] = front_traces.apply(lambda r: extract_downstream(r['trace_name'], r['cmdb_id']), axis=1)

# 3) For each downstream, compute metric_fault_kpi_count and earliest metric anomaly timestamp among those KPIs
# Filter metric_agg to anomaly_count_in_window > 0
metric_faults = metric_agg[metric_agg['anomaly_count_in_window'] > 0].copy()

# Compute per-downstream aggregated metrics
metric_by_comp = metric_faults.groupby('cmdb_id').agg(
    metric_fault_kpi_count=('kpi_name', 'nunique'),
    earliest_metric_anomaly_ts=('_earliest_ts_parsed', 'min')
).reset_index().rename(columns={'cmdb_id':'downstream_component'})

# 4) Join front_traces with metric_by_comp on downstream_component
joined = front_traces.merge(metric_by_comp, on='downstream_component', how='left')

# Keep only rows where metric_fault_kpi_count > 0
joined = joined[joined['metric_fault_kpi_count'].fillna(0) > 0].copy()

# Format trace and metric timestamp fields to ISO strings (Asia/Shanghai) or None
def fmt_ts_field(x):
    if pd.isna(x) or x is pd.NaT:
        return None
    # ensure tz-aware in Asia/Shanghai
    ts = pd.Timestamp(x)
    if ts.tz is None:
        ts = ts.tz_localize(tz)
    else:
        ts = ts.tz_convert(tz)
    return ts.isoformat()

# trace earliest timestamp may be stored as string or Timestamp in trace_agg; parse similarly
joined['trace_earliest_anomaly'] = joined['earliest_anomaly_timestamp_in_window'].apply(lambda x: fmt_ts_field(x))

joined['earliest_metric_anomaly_timestamp'] = joined['earliest_metric_anomaly_ts'].apply(lambda x: fmt_ts_field(x))

# Prepare final compact table with requested columns and renames
final_cols = [
    ('cmdb_id', 'trace_cmdb_id'),
    ('trace_name', 'trace_name'),
    ('p95_threshold', 'trace_p95'),
    ('anomaly_count_in_window', 'trace_anomaly_count'),
    ('trace_earliest_anomaly', 'trace_earliest_anomaly'),
    ('downstream_component', 'downstream_component'),
    ('metric_fault_kpi_count', 'metric_fault_kpi_count'),
    ('earliest_metric_anomaly_timestamp', 'earliest_metric_anomaly_timestamp')
]

final_df = joined[[c for c,_ in final_cols]].copy()
final_df.columns = [new for _,new in final_cols]

# Sort by metric_fault_kpi_count desc, then earliest_metric_anomaly_timestamp asc (nulls last)
final_df['metric_fault_kpi_count'] = final_df['metric_fault_kpi_count'].astype(int)
final_df['earliest_metric_anomaly_timestamp_sort'] = pd.to_datetime(final_df['earliest_metric_anomaly_timestamp']).apply(lambda x: x.tz_localize(tz) if x.tz is None and pd.notna(x) else x)
final_df = final_df.sort_values(['metric_fault_kpi_count', 'earliest_metric_anomaly_timestamp_sort'], ascending=[False, True])

# Limit to top 20 rows for compact display
correlated_table = final_df.loc[:, ['trace_cmdb_id','trace_name','trace_p95','trace_anomaly_count','trace_earliest_anomaly',
                                   'downstream_component','metric_fault_kpi_count','earliest_metric_anomaly_timestamp']].head(20).reset_index(drop=True)

# Keep correlated_table in kernel for follow-up
correlated_table, correlated_table.shape

```
Out[5]:
```
```python
# Compose a compact plain-English summary of the correlation results and display as a single string.
summary_lines = []
summary_lines.append("Correlation summary (frontend-origin traces → metric faults):")
summary_lines.append("- We matched frontend-origin traces (trace.from_frontend*, trace.to_* from frontends) to downstream components' metric anomalies using precomputed aggregations.")
summary_lines.append("- Primary finding: checkoutservice-0 is the dominant downstream hotspot.")
summary_lines.append("  • checkoutservice-0 has 130 distinct metric KPIs with anomalies in the incident window; earliest metric anomaly = 2022-03-21T10:30:00+08:00.")
summary_lines.append("- Frontend-origin traces that map to checkoutservice-0 show anomalies in the same window:")
summary_lines.append("  • Several duration traces (trace.from_frontend-*.duration_mean / duration_p95) show small counts of anomalous samples (trace_anomaly_count ≈ 2) with earliest duration anomalies between 10:36–10:48.")
summary_lines.append("  • Error-rate traces (trace.*.error_rate) show many anomalous samples (trace_anomaly_count ≈ 27–29) with earliest anomalies at 2022-03-21T10:30:00+08:00.")
summary_lines.append("  • Row-count traces show moderate anomaly counts (e.g., 4–8) with earliest anomalies around 10:33–10:48.")
summary_lines.append("- Interpretation: frontend-observed failures/err_rates toward checkoutservice-0 coincide with broad metric-level faults on checkoutservice-0 starting at 10:30+08:00. This indicates a strong correlation and suggests checkoutservice-0 should be investigated as a likely downstream contributor to the frontend errors.")
summary_lines.append("- The correlated table is stored in kernel variable 'correlated_table' for follow-up drilling (top 20 rows saved).")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        trace_cmdb_id                                trace_name  trace_p95  trace_anomaly_count     trace_earliest_anomaly downstream_component  metric_fault_kpi_count earliest_metric_anomaly_timestamp
0   checkoutservice-0       trace.from_frontend-0.duration_mean   0.078738                    2  2022-03-21T10:36:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
1   checkoutservice-0        trace.from_frontend-0.duration_p95   0.078738                    2  2022-03-21T10:36:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
2   checkoutservice-0          trace.from_frontend-0.error_rate   0.000000                   27  2022-03-21T10:30:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
3   checkoutservice-0           trace.from_frontend-0.row_count   2.000000                    8  2022-03-21T10:33:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
4   checkoutservice-0       trace.from_frontend-1.duration_mean   0.071173                    2  2022-03-21T10:45:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
5   checkoutservice-0        trace.from_frontend-1.duration_p95   0.081551                    2  2022-03-21T10:48:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
6   checkoutservice-0          trace.from_frontend-1.error_rate   0.000000                   29  2022-03-21T10:30:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
7   checkoutservice-0           trace.from_frontend-1.row_count   2.000000                    4  2022-03-21T10:48:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
8   checkoutservice-0       trace.from_frontend-2.duration_mean   0.042707                    2  2022-03-21T10:44:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
9   checkoutservice-0        trace.from_frontend-2.duration_p95   0.042745                    2  2022-03-21T10:44:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
10  checkoutservice-0          trace.from_frontend-2.error_rate   0.000000                   28  2022-03-21T10:30:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
11  checkoutservice-0           trace.from_frontend-2.row_count   2.000000                    7  2022-03-21T10:30:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
12         frontend-0  trace.to_checkoutservice-0.duration_mean   0.078738                    2  2022-03-21T10:36:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
13         frontend-0   trace.to_checkoutservice-0.duration_p95   0.078738                    2  2022-03-21T10:36:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
14         frontend-0     trace.to_checkoutservice-0.error_rate   0.000000                   27  2022-03-21T10:30:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
15         frontend-0      trace.to_checkoutservice-0.row_count   2.000000                    8  2022-03-21T10:33:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
16         frontend-1  trace.to_checkoutservice-0.duration_mean   0.071173                    2  2022-03-21T10:45:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
17         frontend-1   trace.to_checkoutservice-0.duration_p95   0.081551                    2  2022-03-21T10:48:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
18         frontend-1     trace.to_checkoutservice-0.error_rate   0.000000                   29  2022-03-21T10:30:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00
19         frontend-1      trace.to_checkoutservice-0.row_count   2.000000                    4  2022-03-21T10:48:00+08:00    checkoutservice-0                     130         2022-03-21T10:30:00+08:00, (20, 8))```
```

In [ ]:
# In[6]:

import pandas as pd

# Reuse agg_metric_p95_window and tz from kernel
df_chk = agg_metric_p95_window.copy()

# Filter for checkoutservice-0
df_chk = df_chk[df_chk['cmdb_id'] == 'checkoutservice-0'].copy()

# Helper to format earliest timestamp to Asia/Shanghai ISO or None
def fmt_ts(x):
    if pd.isna(x) or x is pd.NaT:
        return None
    ts = pd.Timestamp(x)
    try:
        # if naive, localize to Asia/Shanghai; if tz-aware, convert
        if ts.tz is None:
            ts = ts.tz_localize(tz)
        else:
            ts = ts.tz_convert(tz)
    except Exception:
        # fallback: just return isoformat of naive ts
        pass
    return ts.isoformat()

# Apply formatting
if 'earliest_anomaly_timestamp_in_window' in df_chk.columns:
    df_chk['earliest_anomaly_timestamp_in_window'] = df_chk['earliest_anomaly_timestamp_in_window'].apply(fmt_ts)
else:
    df_chk['earliest_anomaly_timestamp_in_window'] = None

# Select requested columns and sort
out_cols = ['cmdb_id', 'kpi_name', 'p95_threshold', 'total_points_in_window',
            'anomaly_count_in_window', 'earliest_anomaly_timestamp_in_window',
            'max_value_in_window', 'max_over_threshold_pct']

df_chk_out = df_chk.loc[:, out_cols].sort_values(
    ['anomaly_count_in_window', 'max_over_threshold_pct'],
    ascending=[False, False]
).head(20).reset_index(drop=True)

# Count distinct KPI names with anomaly_count_in_window > 0 for checkoutservice-0
distinct_kpi_count = int(df_chk[df_chk['anomaly_count_in_window'] > 0]['kpi_name'].nunique())
summary_line = f"checkoutservice-0 distinct KPI names with anomaly_count_in_window>0: {distinct_kpi_count}"

# Return compact table and one-line count
df_chk_out, summary_line

```
Out[6]:
```
- Filtered agg_metric_p95_window for cmdb_id = "checkoutservice-0" and returned the top 20 KPI rows (sorted by anomaly_count_in_window desc, then max_over_threshold_pct desc).  
- Key observations from the top 20 rows: many container/node resource KPIs (e.g., container_file_descriptors, container_fs_limit_MB, container_memory_cache, container_memory_failcnt, container_memory_mapped_file, container_memory_max_usage_MB, container_sockets, container_spec_cpu_*) show anomaly_count_in_window = 30 (all 30 points in the 30-minute window) with earliest anomaly timestamp = 2022-03-21T10:30:00+08:00. For many of these the max value equals the p95 (max_over_threshold_pct = 0.0).  
- Some KPIs have p95 = 0 leading to NaN for max_over_threshold_pct (not meaningful).  
- Interpretation: checkoutservice-0 exhibits broad, full-window metric anomalies across many resource-related KPIs beginning at the incident window start (2022-03-21T10:30:00+08:00), indicating a widespread metric-level issue on that component.  
- checkoutservice-0 distinct KPI names with anomaly_count_in_window>0: 130.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name  p95_threshold  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  max_over_threshold_pct
0   checkoutservice-0        container.node-5.container_file_descriptors   9.000000e+00                      30                       30            2022-03-21T10:30:00+08:00         9.000000e+00                     0.0
1   checkoutservice-0   container.node-5.container_fs_limit_MB./dev/vda1   6.046307e+05                      30                       30            2022-03-21T10:30:00+08:00         6.046307e+05                     0.0
2   checkoutservice-0            container.node-5.container_memory_cache   1.131725e+07                      30                       30            2022-03-21T10:30:00+08:00         1.131725e+07                     0.0
3   checkoutservice-0          container.node-5.container_memory_failcnt   1.536501e+06                      30                       30            2022-03-21T10:30:00+08:00         1.536501e+06                     0.0
4   checkoutservice-0      container.node-5.container_memory_mapped_file   3.276800e+04                      30                       30            2022-03-21T10:30:00+08:00         3.276800e+04                     0.0
5   checkoutservice-0     container.node-5.container_memory_max_usage_MB   1.280000e+02                      30                       30            2022-03-21T10:30:00+08:00         1.280000e+02                     0.0
6   checkoutservice-0                 container.node-5.container_sockets   3.000000e+00                      30                       30            2022-03-21T10:30:00+08:00         3.000000e+00                     0.0
7   checkoutservice-0         container.node-5.container_spec_cpu_period   1.000000e+05                      30                       30            2022-03-21T10:30:00+08:00         1.000000e+05                     0.0
8   checkoutservice-0          container.node-5.container_spec_cpu_quota   2.000000e+04                      30                       30            2022-03-21T10:30:00+08:00         2.000000e+04                     0.0
9   checkoutservice-0         container.node-5.container_spec_cpu_shares   1.020000e+02                      30                       30            2022-03-21T10:30:00+08:00         1.020000e+02                     0.0
10  checkoutservice-0    container.node-5.container_spec_memory_limit_MB   1.280000e+02                      30                       30            2022-03-21T10:30:00+08:00         1.280000e+02                     0.0
11  checkoutservice-0  container.node-5.container_spec_memory_swap_li...   1.280000e+02                      30                       30            2022-03-21T10:30:00+08:00         1.280000e+02                     0.0
12  checkoutservice-0      container.node-5.container_start_time_seconds   1.647523e+09                      30                       30            2022-03-21T10:30:00+08:00         1.647523e+09                     0.0
13  checkoutservice-0                 container.node-5.container_threads   2.100000e+01                      30                       30            2022-03-21T10:30:00+08:00         2.100000e+01                     0.0
14  checkoutservice-0  container.node-5.container_ulimits_soft.max_op...   6.553600e+04                      30                       30            2022-03-21T10:30:00+08:00         6.553600e+04                     0.0
15  checkoutservice-0    container.node-5.container_cpu_load_average_10s   0.000000e+00                      30                       30            2022-03-21T10:30:00+08:00         0.000000e+00                     NaN
16  checkoutservice-0     container.node-5.container_fs_inodes./dev/vda1   0.000000e+00                      30                       30            2022-03-21T10:30:00+08:00         0.000000e+00                     NaN
17  checkoutservice-0  container.node-5.container_fs_inodes_free./dev...   0.000000e+00                      30                       30            2022-03-21T10:30:00+08:00         0.000000e+00                     NaN
18  checkoutservice-0  container.node-5.container_fs_io_current./dev/...   0.000000e+00                      30                       30            2022-03-21T10:30:00+08:00         0.000000e+00                     NaN
19  checkoutservice-0  container.node-5.container_fs_io_time_seconds....   0.000000e+00                      30                       30            2022-03-21T10:30:00+08:00         0.000000e+00                     NaN, 'checkoutservice-0 distinct KPI names with anomaly_count_in_window>0: 130')```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: agg_metric_p95_window, df_error, tz if available
# If tz not in kernel, define it
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# 1) KPI top 10 for checkoutservice-0 by max_over_threshold_pct desc
checkout_kpis = agg_metric_p95_window[agg_metric_p95_window['cmdb_id'] == 'checkoutservice-0'].copy()

# Helper to format timestamps to Asia/Shanghai ISO or None
def fmt_ts(x):
    if pd.isna(x) or x is pd.NaT:
        return None
    ts = pd.Timestamp(x)
    # if tz-naive, localize to Asia/Shanghai; if tz-aware, convert
    if ts.tz is None:
        ts = ts.tz_localize(tz)
    else:
        ts = ts.tz_convert(tz)
    return ts.isoformat()

# Format earliest_anomaly_timestamp_in_window for display
if 'earliest_anomaly_timestamp_in_window' in checkout_kpis.columns:
    checkout_kpis['earliest_anomaly_timestamp_in_window'] = checkout_kpis['earliest_anomaly_timestamp_in_window'].apply(fmt_ts)
else:
    checkout_kpis['earliest_anomaly_timestamp_in_window'] = None

# Select requested columns and sort by max_over_threshold_pct desc (NaNs last)
cols_out = ['kpi_name', 'p95_threshold', 'anomaly_count_in_window',
            'earliest_anomaly_timestamp_in_window', 'max_value_in_window', 'max_over_threshold_pct']

checkout_kpis_top10 = checkout_kpis.loc[:, cols_out].sort_values(
    by='max_over_threshold_pct', ascending=False, na_position='last'
).head(10).reset_index(drop=True)

# Count distinct KPI names with anomaly_count_in_window > 0
checkout_kpi_count = int(checkout_kpis[checkout_kpis['anomaly_count_in_window'] > 0]['kpi_name'].nunique())
checkout_kpi_count_line = f"checkoutservice-0 distinct KPI names with anomaly_count_in_window>0: {checkout_kpi_count}"

# Store results in kernel variables for follow-up
checkout_kpi_top10 = checkout_kpis_top10
checkout_kpi_count_str = checkout_kpi_count_line

# 2) Error logs for checkoutservice-0 within incident window
start = pd.Timestamp('2022-03-21T10:30:00', tz=tz)
end = pd.Timestamp('2022-03-21T11:00:00', tz=tz)

# Use df_error from kernel
err_window = df_error[(df_error['_ts_sh'] >= start) & (df_error['_ts_sh'] < end) & (df_error['cmdb_id'] == 'checkoutservice-0')].copy()

# Aggregates
message_count_in_window = int(len(err_window))
if message_count_in_window > 0:
    earliest_msg_ts = err_window['_ts_sh'].min()
    earliest_msg_iso = fmt_ts(earliest_msg_ts)
else:
    earliest_msg_iso = None

# Up to 10 sample messages sorted by timestamp
err_samples = err_window.sort_values('timestamp', ascending=True).head(10).copy()
if not err_samples.empty:
    err_samples['timestamp_Asia/Shanghai'] = err_samples['_ts_sh'].apply(fmt_ts)
    err_samples['message_sample'] = err_samples['message'].astype(str).str.slice(0, 300)
    checkout_error_samples = err_samples.loc[:, ['timestamp_Asia/Shanghai', 'message_sample']].reset_index(drop=True)
else:
    checkout_error_samples = pd.DataFrame(columns=['timestamp_Asia/Shanghai', 'message_sample'])

# Store summary in kernel variable
checkout_error_summary = pd.DataFrame([{
    'cmdb_id': 'checkoutservice-0',
    'message_count_in_window': message_count_in_window,
    'earliest_message_timestamp_in_window': earliest_msg_iso
}])

# Return compact results (DataFrames / strings), stored variables retained for follow-up
checkout_kpi_top10, checkout_kpi_count_str, checkout_error_summary, checkout_error_samples

```
Out[7]:
```
```python
# Compose a compact plain-English summary string for the requested results and display it.
import pandas as pd
import numpy as np

lines = []
lines.append("Summary for checkoutservice-0 during 2022-03-21T10:30:00+08:00 to 2022-03-21T11:00:00+08:00:")

# Part 1: Top 10 KPI rows by max_over_threshold_pct
lines.append("")
lines.append("1) Top 10 KPIs (by max_over_threshold_pct):")
if 'checkout_kpi_top10' in globals() and not checkout_kpi_top10.empty:
    for _, r in checkout_kpi_top10.iterrows():
        k = r['kpi_name']
        p95 = r['p95_threshold']
        anom = int(r['anomaly_count_in_window']) if not pd.isna(r['anomaly_count_in_window']) else 0
        earliest = r['earliest_anomaly_timestamp_in_window'] if pd.notna(r['earliest_anomaly_timestamp_in_window']) else "None"
        mv = r['max_value_in_window']
        pct = r['max_over_threshold_pct']
        p95_s = f"{p95:.6g}" if pd.notna(p95) else "NaN"
        mv_s = f"{mv:.6g}" if pd.notna(mv) else "NaN"
        pct_s = f"{pct:.2f}%" if pd.notna(pct) else "NaN"
        lines.append(f"  - {k} | p95={p95_s} | anomalies={anom} | earliest={earliest} | max={mv_s} | over={pct_s}")
else:
    lines.append("  (no KPI rows found)")

# One-line count of distinct KPI names with anomalies
if 'checkout_kpi_count_str' in globals():
    lines.append("")
    lines.append(f"  {checkout_kpi_count_str}")
else:
    lines.append("")
    lines.append("  checkoutservice-0 distinct KPI names with anomaly_count_in_window>0: (unknown)")

# Part 2: Error logs for checkoutservice-0 in the window
lines.append("")
lines.append("2) Error logs for checkoutservice-0 in-window:")
if 'checkout_error_summary' in globals() and not checkout_error_summary.empty:
    row = checkout_error_summary.iloc[0]
    cnt = int(row['message_count_in_window'])
    earliest = row['earliest_message_timestamp_in_window'] if pd.notna(row['earliest_message_timestamp_in_window']) else None
    lines.append(f"  message_count_in_window = {cnt}, earliest_message_timestamp_in_window = {earliest}")
    if cnt > 0 and 'checkout_error_samples' in globals() and not checkout_error_samples.empty:
        lines.append("  Up to 10 sample messages:")
        for _, s in checkout_error_samples.iterrows():
            ts = s['timestamp_Asia/Shanghai']
            msg = s['message_sample']
            lines.append(f"    {ts} - {msg}")
    else:
        lines.append("  No sample error messages (count = 0).")
else:
    lines.append("  No error log summary available.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                            kpi_name  p95_threshold  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  max_over_threshold_pct
0  container.node-5.container_network_receive_MB....       0.193949                        2            2022-03-21T10:34:00+08:00             0.308527               59.076079
1  container.node-5.container_cpu_cfs_throttled_s...       0.232365                        2            2022-03-21T10:50:00+08:00             0.334723               44.050207
2      container.node-5.container_cpu_system_seconds       0.037750                        2            2022-03-21T10:41:00+08:00             0.050000               32.450331
3       container.node-5.container_cpu_usage_seconds       0.117600                        2            2022-03-21T10:47:00+08:00             0.133418               13.450260
4        container.node-5.container_cpu_user_seconds       0.067750                        2            2022-03-21T10:47:00+08:00             0.075000               10.701107
5  mesh.destination.frontend.checkoutservice.isti...       0.191325                        2            2022-03-21T10:45:00+08:00             0.210500               10.022214
6         container.node-5.container_cpu_cfs_periods      22.275000                        2            2022-03-21T10:47:00+08:00            24.000000                7.744108
7  container.node-5.container_cpu_cfs_throttled_p...       3.275000                        2            2022-03-21T10:50:00+08:00             3.500000                6.870229
8  container.node-5.container_memory_failures.con...     975.775000                        2            2022-03-21T10:58:00+08:00          1026.000000                5.147191
9  container.node-5.container_memory_failures.hie...     975.775000                        2            2022-03-21T10:58:00+08:00          1026.000000                5.147191, 'checkoutservice-0 distinct KPI names with anomaly_count_in_window>0: 130',              cmdb_id  message_count_in_window earliest_message_timestamp_in_window
0  checkoutservice-0                        0                                 None, Empty DataFrame
Columns: [timestamp_Asia/Shanghai, message_sample]
Index: [])```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse agg_metric_p95_window from kernel
try:
    metric_agg = agg_metric_p95_window.copy()
except NameError:
    raise RuntimeError("agg_metric_p95_window not found in kernel.")

# Ensure timezone
tz = pytz.timezone('Asia/Shanghai')

# Helper to parse/normalize earliest timestamp values to tz-aware pandas.Timestamp
def parse_to_shanghai(x):
    if pd.isna(x):
        return pd.NaT
    try:
        ts = pd.to_datetime(x)
    except Exception:
        return pd.NaT
    # If already tz-aware, convert; if naive, localize to Asia/Shanghai
    try:
        if ts.tz is None:
            ts = ts.tz_localize(tz)
        else:
            ts = ts.tz_convert(tz)
    except Exception:
        # fallback
        try:
            ts = ts.tz_localize(tz)
        except Exception:
            return pd.NaT
    return ts

# Prepare a parsed earliest timestamp column for reliable min computation
if 'earliest_anomaly_timestamp_in_window' in metric_agg.columns:
    metric_agg['_earliest_ts_parsed'] = metric_agg['earliest_anomaly_timestamp_in_window'].apply(parse_to_shanghai)
else:
    metric_agg['_earliest_ts_parsed'] = pd.NaT

# Ensure anomaly_count_in_window is numeric (fill missing with 0)
metric_agg['anomaly_count_in_window'] = pd.to_numeric(metric_agg['anomaly_count_in_window'], errors='coerce').fillna(0).astype(int)

rows = []
for node_label in ['node-5', 'node-6']:
    # Select rows where kpi_name contains the node label
    sel = metric_agg[metric_agg['kpi_name'].str.contains(node_label, na=False)].copy()
    # Keep only anomalous rows (anomaly_count_in_window > 0)
    sel_anom = sel[sel['anomaly_count_in_window'] > 0]
    distinct_affected_cmdb_count = int(sel_anom['cmdb_id'].nunique())
    total_anomalous_kpi_count = int(sel_anom.shape[0])
    # earliest anomaly timestamp among those rows
    if not sel_anom.empty:
        earliest_ts = sel_anom['_earliest_ts_parsed'].min()
        earliest_iso = earliest_ts.isoformat() if pd.notna(earliest_ts) else None
    else:
        earliest_iso = None
    rows.append({
        'node_label': node_label,
        'distinct_affected_cmdb_count': distinct_affected_cmdb_count,
        'total_anomalous_kpi_count': total_anomalous_kpi_count,
        'earliest_anomaly_timestamp_in_window': earliest_iso
    })

node5_node6_agg = pd.DataFrame(rows).sort_values('total_anomalous_kpi_count', ascending=False).reset_index(drop=True)

# Return the compact table (stored in kernel variable 'node5_node6_agg')
node5_node6_agg

```
Out[8]:
```
- node-5: 35 distinct affected components, 2,124 anomalous KPI rows in the incident window; earliest metric anomaly = 2022-03-21T10:30:00+08:00.  
- node-6: 7 distinct affected components, 448 anomalous KPI rows in the incident window; earliest metric anomaly = 2022-03-21T10:30:00+08:00.  
- Conclusion: node-5 is far more impacted than node-6 (many more affected components and anomalous KPIs), and anomalies on both nodes begin at the incident window start (10:30+08:00).

The original code execution output of IPython Kernel is also provided below for reference:

node_label  distinct_affected_cmdb_count  total_anomalous_kpi_count earliest_anomaly_timestamp_in_window
0     node-5                            35                       2124            2022-03-21T10:30:00+08:00
1     node-6                             7                        448            2022-03-21T10:30:00+08:00```
```